In [3]:
#Use Python 3.12.x
import torch

In [4]:
import os
import math
import copy
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# PyTorch Geometric for GNN structures
try:
    from torch_geometric.nn import GCNConv, GATConv
except ImportError:
    raise ImportError("Please run: pip install torch-geometric")

# KAN Layer
try:
    from efficient_kan import KAN
except ImportError:
    raise ImportError("Please run: pip install efficient-kan")

# ─────────────────────────────────────────────────────────────────
# 1. GLOBAL PIPELINE CONFIGURATION
# ─────────────────────────────────────────────────────────────────

CSV_PATH        = "/content/final_data.npz"
DATE_FORMAT     = "%d-%m-%Y %H:%M"

LOOKBACK        = 12
FORECAST_STEPS  = 5
BATCH_SIZE      = 64
LR              = 1e-3
WEIGHT_DECAY    = 1e-4

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

# Core Architecture Hidden Sizes
GNN_HIDDEN   = 32
LSTM_HIDDEN  = 128
LSTM_LAYERS  = 2
LSTM_DROPOUT = 0.2

# Increased grid size for better spline resolution and variance mapping
KAN_GRID_SIZE    = 8
KAN_SPLINE_ORDER = 3

SENSORS = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- STRICT COMPULSORY FEDERATED CONFIGURATION ---
FED_ROUNDS   = 100
LOCAL_EPOCHS = 3
NUM_CLIENTS  = 4


# ─────────────────────────────────────────────────────────────────
# 2. RUNTIME LOGGERS & TRACKING ENGINE
# ─────────────────────────────────────────────────────────────────

class FederatedPerformanceTracker:
    """Tracks validation history metrics across global federated rounds."""
    def __init__(self):
        self.history = {
            "round": [], "val_huber": [], "val_mse": [], "val_rmse": []
        }

    def update(self, rnd, vl_h, vl_m):
        self.history["round"].append(rnd)
        self.history["val_huber"].append(vl_h)
        self.history["val_mse"].append(vl_m)
        self.history["val_rmse"].append(math.sqrt(vl_m))


def log_federated_header(model_name: str):
    print(f"\n{'='*70}")
    print(f" INITIALIZING FEDERATED EXECUTIVE RUN: [ {model_name} ]")
    print(f"{'='*70}")
    print(f" Target Compute Platform : {DEVICE}")
    print(f" Federated Global Rounds : {FED_ROUNDS}")
    print(f" Local Client Epochs     : {LOCAL_EPOCHS}")
    print(f" Total Edge Clients      : {NUM_CLIENTS}")
    print(f"─"*70 + "\n")


# ─────────────────────────────────────────────────────────────────
# 3. SPATIAL GEOMETRY GRAPH CONFIGURATION
# ─────────────────────────────────────────────────────────────────

def build_spatial_graph(sensor_names):
    num_nodes = len(sensor_names)
    edges_src = list(range(num_nodes))
    edges_dst = list(range(num_nodes))

    for i in range(num_nodes - 1):
        edges_src.extend([i, i + 1])
        edges_dst.extend([i + 1, i])

    edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
    return edge_index


# ─────────────────────────────────────────────────────────────────
# 4. DATA LOADER STRATEGIES & SIMULATED DISTRIBUTION
# ─────────────────────────────────────────────────────────────────

def load_csv(path: str) -> pd.DataFrame:
    if path.endswith('.npz'):
        archive = np.load(path, allow_pickle=True)
        raw_data = archive['data'] if 'data' in archive.files else archive[archive.files[0]]
        indices  = archive['index'] if 'index' in archive.files else None
        cols     = archive['columns'] if 'columns' in archive.files else None

        if raw_data.ndim == 3:
            raw_data = raw_data[:, :, 0]

        if indices is None:
            indices = pd.date_range("2019-01-01 00:00", periods=len(raw_data), freq="15min")
        if cols is None:
            cols = [f"sensor_{i}" for i in range(raw_data.shape[1])]

        df = pd.DataFrame(raw_data, index=indices, columns=cols)
    else:
        df = pd.read_csv(path, index_col=0)
        df.index = pd.to_datetime(df.index, format=DATE_FORMAT)

    df = df.sort_index().apply(pd.to_numeric, errors="coerce")
    df = df.ffill().bfill()
    return df


def make_synthetic() -> pd.DataFrame:
    rng = np.random.default_rng(42)
    T, S = 1200, 5
    t = np.linspace(0, 4 * np.pi, T)
    data = {}

    for s in range(S):
        wave = 50 * np.sin(t + s * 0.5)
        noise = rng.normal(0, 4, T)
        data[f"node_{s:02d}"] = np.clip(wave + noise + 65, 0, 130).astype(int)

    return pd.DataFrame(data, index=pd.date_range("2019-01-01 00:00", periods=T, freq="15min"))


class SpatioTemporalDataset(Dataset):
    def __init__(self, arr: np.ndarray):
        xs, ys = [], []
        n = len(arr)

        for i in range(n - LOOKBACK - FORECAST_STEPS + 1):
            xs.append(np.expand_dims(arr[i : i + LOOKBACK], axis=-1))
            ys.append(arr[i + LOOKBACK : i + LOOKBACK + FORECAST_STEPS].flatten())

        self.X = torch.tensor(np.array(xs), dtype=torch.float32)
        self.y = torch.tensor(np.array(ys), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def split_dataset_to_clients(dataset, num_clients):
    """Slices centralized data arrays into local subsets for client simulation."""
    total_size = len(dataset)
    base_size = total_size // num_clients
    client_loaders = []

    for c in range(num_clients):
        start_idx = c * base_size
        end_idx = total_size if c == num_clients - 1 else (c + 1) * base_size
        indices = list(range(start_idx, end_idx))

        subset = Subset(dataset, indices)
        loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)
        client_loaders.append(loader)

    return client_loaders


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_active_parameters(model):
    """Calculates both total parameters and non-zero active parameters to track sparsity."""
    total_params = 0
    non_zero_params = 0
    for name, p in model.named_parameters():
        if p.requires_grad:
            total_params += p.numel()
            non_zero_params += (p != 0).sum().item()
    return total_params, non_zero_params


def apply_global_pruning(model, amount=0.30):
    """Applies global unstructured pruning explicitly targeting standard linear and recurrent structures safely."""
    parameters_to_prune = []
    for module in model.modules():
        # Only target standard feedforward and recurrent structures to avoid PyG internal name divergence
        if isinstance(module, (nn.Linear, nn.GRU)):
            for name, param in module.named_parameters():
                if 'weight' in name:
                    parameters_to_prune.append((module, name))

    if parameters_to_prune:
        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=amount,
        )
        for module, name in parameters_to_prune:
            prune.remove(module, name)


# ─────────────────────────────────────────────────────────────────
# 5. OPTIMIZED MODEL ARCHITECTURES FOR FEDAVG CONSTRAINTS
# ─────────────────────────────────────────────────────────────────

class GNNGRUKANForecaster(nn.Module):
    def __init__(self, n_sensors: int, edge_index: torch.Tensor):
        super().__init__()
        self.n_sensors = n_sensors
        self.register_buffer("edge_index", edge_index)
        out_dim = FORECAST_STEPS * n_sensors

        self.gat_layer = GATConv(in_channels=1, out_channels=16, heads=2, concat=True)
        self.gcn_layer = GCNConv(in_channels=32, out_channels=GNN_HIDDEN)
        self.spatial_act = nn.SiLU()

        self.gru = nn.GRU(
            input_size=n_sensors * GNN_HIDDEN,
            hidden_size=LSTM_HIDDEN,
            num_layers=LSTM_LAYERS,
            batch_first=True,
            dropout=LSTM_DROPOUT if LSTM_LAYERS > 1 else 0.0
        )
        self.drop = nn.Dropout(p=0.1)
        self.norm_layer = nn.LayerNorm(LSTM_HIDDEN)

        h1 = min(max(LSTM_HIDDEN, 64), 256)
        h2 = max(LSTM_HIDDEN // 2, 32)

        self.kan = KAN(
            layers_hidden=[LSTM_HIDDEN, h1, h2, out_dim],
            grid_size=KAN_GRID_SIZE,
            spline_order=KAN_SPLINE_ORDER
        )

    def forward(self, x):
        batch_size, lookback_len, n_nodes, n_feats = x.shape
        spatial_steps = []

        for t in range(lookback_len):
            snapshot = x[:, t, :, :].reshape(-1, n_feats)
            shifted_edges = self._get_batch_edges(batch_size, n_nodes)

            feat_map = self.gat_layer(snapshot, shifted_edges)
            feat_map = self.spatial_act(feat_map)
            emb = self.spatial_act(self.gcn_layer(feat_map, shifted_edges))
            spatial_steps.append(emb.reshape(batch_size, n_nodes * GNN_HIDDEN))

        _, h_n = self.gru(torch.stack(spatial_steps, dim=1))
        stabilized_features = self.norm_layer(h_n[-1])
        return self.kan(self.drop(stabilized_features))

    def _get_batch_edges(self, batch_size, n_nodes):
        if batch_size == 1:
            return self.edge_index
        return torch.cat([self.edge_index + (b * n_nodes) for b in range(batch_size)], dim=1)


class STGAGRUGCNForecaster(nn.Module):
    def __init__(self, n_sensors: int, edge_index: torch.Tensor):
        super().__init__()
        self.n_sensors = n_sensors
        self.register_buffer("edge_index", edge_index)
        out_dim = FORECAST_STEPS * n_sensors

        self.gat_layer = GATConv(in_channels=1, out_channels=16, heads=2, concat=True)
        self.gcn_layer = GCNConv(in_channels=32, out_channels=GNN_HIDDEN)
        self.spatial_act = nn.SiLU()

        self.temporal_regressor = nn.GRU(
            input_size=n_sensors * GNN_HIDDEN,
            hidden_size=LSTM_HIDDEN,
            num_layers=LSTM_LAYERS,
            batch_first=True,
            dropout=LSTM_DROPOUT
        )
        self.norm_layer = nn.LayerNorm(LSTM_HIDDEN)

        self.fc_head = nn.Sequential(
            nn.Linear(LSTM_HIDDEN, LSTM_HIDDEN),
            nn.SiLU(),
            nn.Dropout(p=0.1),
            nn.Linear(LSTM_HIDDEN, out_dim)
        )

    def forward(self, x):
        batch_size, lookback_len, n_nodes, n_feats = x.shape
        spatial_steps = []

        for t in range(lookback_len):
            snapshot = x[:, t, :, :].reshape(-1, n_feats)
            shifted_edges = self._get_batch_edges(batch_size, n_nodes)
            feat_map = self.gat_layer(snapshot, shifted_edges)
            feat_map = self.spatial_act(feat_map)
            emb = self.spatial_act(self.gcn_layer(feat_map, shifted_edges))
            spatial_steps.append(emb.reshape(batch_size, n_nodes * GNN_HIDDEN))

        temporal_tensor = torch.stack(spatial_steps, dim=1)
        _, h_n = self.temporal_regressor(temporal_tensor)

        stabilized_features = self.norm_layer(h_n[-1])
        return self.fc_head(stabilized_features)

    def _get_batch_edges(self, batch_size, n_nodes):
        if batch_size == 1:
            return self.edge_index
        return torch.cat([self.edge_index + (b * n_nodes) for b in range(batch_size)], dim=1)


# ─────────────────────────────────────────────────────────────────
# 6. COMPULSORY FEDERATED AVERAGING ENGINE METHODS
# ─────────────────────────────────────────────────────────────────

def local_train_client(global_model_state, client_loader, rnd):
    local_model = copy.deepcopy(global_model_state)
    local_model.train()

    optimizer = torch.optim.AdamW(local_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.HuberLoss(delta=1.0)

    for epoch in range(LOCAL_EPOCHS):
        for xb, yb in client_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            preds = local_model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
            optimizer.step()

    return local_model.state_dict(), len(client_loader.dataset)


def federated_averaging(global_model, local_states, total_samples):
    global_state = global_model.state_dict()
    fed_avg_state = copy.deepcopy(global_state)

    for key in fed_avg_state.keys():
        if fed_avg_state[key].dtype == torch.long or "edge_index" in key:
            continue
        fed_avg_state[key] = torch.zeros_like(fed_avg_state[key], dtype=torch.float32)

    for key in fed_avg_state.keys():
        if fed_avg_state[key].dtype == torch.long or "edge_index" in key:
            continue
        for local_state, sample_count in local_states:
            weight_factor = sample_count / total_samples
            fed_avg_state[key] += local_state[key].to(DEVICE) * weight_factor

    global_model.load_state_dict(fed_avg_state)
    return global_model


def federated_server_engine(global_model, client_loaders, val_dl):
    loss_fn = nn.HuberLoss(delta=1.0)
    mse_fn = nn.MSELoss()
    tracker = FederatedPerformanceTracker()

    print(f"  {'Fed Round':<12} | {'Val Huber Loss':<16} | {'Val MSE Score':<16}")
    print(f"  {'─'*55}")

    for rnd in range(1, FED_ROUNDS + 1):
        local_states = []
        total_samples = 0

        for client_loader in client_loaders:
            local_state, sample_count = local_train_client(global_model, client_loader, rnd)
            local_states.append((local_state, sample_count))
            total_samples += sample_count

        global_model = federated_averaging(global_model, local_states, total_samples)
        global_model.eval()
        v_huber, v_mse = 0.0, 0.0

        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                p = global_model(xb)
                v_huber += loss_fn(p, yb).item() * len(xb)
                v_mse += mse_fn(p, yb).item() * len(xb)

        vl_h = v_huber / len(val_dl.dataset)
        vl_m = v_mse / len(val_dl.dataset)
        tracker.update(rnd, vl_h, vl_m)

        if rnd % 10 == 0 or rnd == 1:
            print(f"  Round {rnd:3d}/{FED_ROUNDS:3d} | {vl_h:14.5f} | {vl_m:14.5f}")

    print(f"  {'─'*55}")
    return tracker


def evaluate_engine(model, test_dl, scaler, n_sensors):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for xb, yb in test_dl:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
            targets.append(yb.numpy())

    P = np.concatenate(preds)
    T = np.concatenate(targets)

    def invert(arr):
        return scaler.inverse_transform(arr.reshape(-1, n_sensors)).reshape(len(arr), FORECAST_STEPS, n_sensors)

    P_inv = invert(P)
    T_inv = invert(T)

    mae = mean_absolute_error(T_inv.flatten(), P_inv.flatten())
    mse = mean_squared_error(T_inv.flatten(), P_inv.flatten())
    rmse = math.sqrt(mse)
    mape = np.mean(np.abs((T_inv - P_inv) / np.clip(np.abs(T_inv), 1e-5, None))) * 100
    smape = 100 * np.mean(2 * np.abs(P_inv - T_inv) / (np.abs(T_inv) + np.abs(P_inv) + 1e-5))
    r2 = r2_score(T_inv.flatten(), P_inv.flatten())

    return mae, mse, rmse, mape, smape, r2


# ─────────────────────────────────────────────────────────────────
# 7. MAIN RUNTIME PROGRAM
# ─────────────────────────────────────────────────────────────────

def main():
    df = load_csv(CSV_PATH) if os.path.exists(CSV_PATH) else make_synthetic()
    sensor_names = df.columns.tolist()
    n_sensors = len(sensor_names)
    edge_index = build_spatial_graph(sensor_names)

    T_len = len(df)
    tr_lim = int(T_len * TRAIN_FRAC)
    val_lim = int(T_len * (TRAIN_FRAC + VAL_FRAC))

    train_raw = df.values[:tr_lim].astype(np.float32)
    val_raw   = df.values[tr_lim:val_lim].astype(np.float32)
    test_raw  = df.values[val_lim:].astype(np.float32)

    scaler = MinMaxScaler()
    scaler.fit(train_raw)

    train_scaled = scaler.transform(train_raw)
    val_scaled   = scaler.transform(val_raw)
    test_scaled  = scaler.transform(test_raw)

    train_dataset = SpatioTemporalDataset(train_scaled)
    client_loaders = split_dataset_to_clients(train_dataset, NUM_CLIENTS)

    val_dl = DataLoader(SpatioTemporalDataset(val_scaled), batch_size=BATCH_SIZE, shuffle=False)
    test_dl = DataLoader(SpatioTemporalDataset(test_scaled), batch_size=BATCH_SIZE, shuffle=False)

    # --- EXECUTION RUN 1: GNN-LSTM-KAN ---
    log_federated_header("GNN-GRU-KAN Federated Engine")
    model_kan = GNNGRUKANForecaster(n_sensors, edge_index).to(DEVICE)

    tot_kan_before, act_kan_before = count_active_parameters(model_kan)
    print(f"[PARAM LOG] GNN-GRU-KAN Pre-Train -> Total Registered: {tot_kan_before}, Active Non-Zero: {act_kan_before}")

    tracker_kan = federated_server_engine(model_kan, client_loaders, val_dl)
    mae_kan, mse_kan, rmse_kan, mape_kan, smape_kan, r2_kan = evaluate_engine(model_kan, test_dl, scaler, n_sensors)

    apply_global_pruning(model_kan, amount=0.30)

    tot_kan_after, act_kan_after = count_active_parameters(model_kan)
    kan_reduction = ((act_kan_before - act_kan_after) / act_kan_before) * 100
    print(f"[PARAM LOG] GNN-GRU-KAN Post-Train -> Total Registered: {tot_kan_after}, Active Non-Zero: {act_kan_after}")
    print(f"[PARAM LOG] GNN-GRU-KAN Active Parameter Footprint Reduction: {kan_reduction:.4f}%\n")

    # --- EXECUTION RUN 2: STGAT + GCN ---
    log_federated_header("STGA-GRU + GCN Federated Engine")
    model_stgat = STGAGRUGCNForecaster(n_sensors, edge_index).to(DEVICE)

    tot_stgat_before, act_stgat_before = count_active_parameters(model_stgat)
    print(f"[PARAM LOG] STGA-GRU+GCN Pre-Train -> Total Registered: {tot_stgat_before}, Active Non-Zero: {act_stgat_before}")

    tracker_stgat = federated_server_engine(model_stgat, client_loaders, val_dl)
    mae_stgat, mse_stgat, rmse_stgat, mape_stgat, smape_stgat, r2_stgat = evaluate_engine(model_stgat, test_dl, scaler, n_sensors)

    apply_global_pruning(model_stgat, amount=0.30)

    tot_stgat_after, act_stgat_after = count_active_parameters(model_stgat)
    stgat_reduction = ((act_stgat_before - act_stgat_after) / act_stgat_before) * 100
    print(f"[PARAM LOG] STGA-GRU+GCN Post-Train -> Total Registered: {tot_stgat_after}, Active Non-Zero: {act_stgat_after}")
    print(f"[PARAM LOG] STGA-GRU+GCN Active Parameter Footprint Reduction: {stgat_reduction:.4f}%\n")

    # --- POST EXECUTION MODEL SIZE AUDIT ---
    print(f"\n{'='*70}")
    print("                  TRAINABLE FOOTPRINT AUDIT SUMMARY")
    print(f"{'='*70}")
    print(f"Total Trainable Parameters in GNN-GRU-KAN: {count_parameters(model_kan)}")
    print(f"Total Trainable Parameters in STGA-GRU + GCN: {count_parameters(model_stgat)}")
    print(f"{'='*70}\n")

    # ─────────────────────────────────────────────────────────────────
    # 8. METRIC INSIGHTS & COMPARATIVE ANALYTICS
    # ─────────────────────────────────────────────────────────────────

    comparison_summary_df = pd.DataFrame({
        "Model Topology Configuration": [
            "GNN-GRU-KAN Federated (FedAvg)",
            "STGA-GRU + GCN Federated (FedAvg)"
        ],
        "Test MAE Loss Score": [mae_kan, mae_stgat],
        "Test MSE Raw Score": [mse_kan, mse_stgat],
        "Test RMSE Score": [rmse_kan, rmse_stgat],
        "Value MAPE %": [mape_kan, mape_stgat],
        "Value SMAPE %": [smape_kan, smape_stgat],
        "R2 Score Metric": [r2_kan, r2_stgat]
    }).set_index("Model Topology Configuration")

    print(f"\n{'='*90}")
    print("                FEDERATED ARCHITECTURE COMPARATIVE MATRIX")
    print(f"{'='*90}")
    print(comparison_summary_df.round(4).to_string())
    print(f"{'─'*90}")
    print(f"{'='*90}\n")


if __name__ == "__main__":
    main()


 INITIALIZING FEDERATED EXECUTIVE RUN: [ GNN-GRU-KAN Federated Engine ]
 Target Compute Platform : cpu
 Federated Global Rounds : 100
 Local Client Epochs     : 3
 Total Edge Clients      : 4
──────────────────────────────────────────────────────────────────────

[PARAM LOG] GNN-GRU-KAN Pre-Train -> Total Registered: 552160, Active Non-Zero: 551968
  Fed Round    | Val Huber Loss   | Val MSE Score   
  ───────────────────────────────────────────────────────
  Round   1/100 |        0.05300 |        0.10600
  Round  10/100 |        0.00169 |        0.00338
  Round  20/100 |        0.00149 |        0.00299
  Round  30/100 |        0.00121 |        0.00243
  Round  40/100 |        0.00102 |        0.00203
  Round  50/100 |        0.00129 |        0.00259
  Round  60/100 |        0.00116 |        0.00232
  Round  70/100 |        0.00122 |        0.00244
  Round  80/100 |        0.00108 |        0.00217
  Round  90/100 |        0.00094 |        0.00189
  Round 100/100 |        0.00098 |   